# IND320 Project P1

Author: Robin Torjussen Egede  
[Streamlit P2 page](https://ind320-hush-p2.streamlit.app) - [P1 page](https://ind320-hush.streamlit.app)  
[Repo & P2 branch](https://github.com/Kunii/IND320-hush/tree/work-p2)  

## Log

- Created snapshot branch of P1 [(found here)](https://github.com/Kunii/IND320-hush/tree/snapshot-p1).
- Created new work branch for P2.
- Reorganized 'loose' P1 files into 'CA1' dirs.
- Added streamlit link to readme.
- Updated package deps (pyspark, pymongo etc)

## AI-usage

Used as "fancy auto-complete" in-line suggestions, docstring & in-line comment drafts, plotting drafts.  
Also used for general idea & "best practice" for the secrets stuff.  
All logic and program flow was strictly concocted by me.

In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

from cassandra.util import datetime_from_uuid1
from cassandra.cluster import Cluster
from pyspark.sql import SparkSession
from pymongo import MongoClient
from src.utils.env import SecretManager

import pandas as pd
import matplotlib.pyplot as plt
import requests

No secrets path provided
Defaulting to: [REDACTED]/IND320-hush/.secrets/secrets.toml


In [2]:
#docker run --name ind320-cassandra -d -p 9042:9042 cassandra:latest
#docker start ind320-cassandra

# Connecting to Cassandra

cluster = Cluster(['localhost'], port=9042)
session = cluster.connect()

db_keyspace = "ind320"
db_table = "testing_table"

session.execute(f"CREATE KEYSPACE IF NOT EXISTS {db_keyspace} WITH REPLICATION = {{ 'class': 'SimpleStrategy', 'replication_factor': 1 }};")
session.set_keyspace(db_keyspace)

session.execute(f"DROP TABLE IF EXISTS {db_table};")

session.execute(f"""CREATE TABLE IF NOT EXISTS {db_table} (
    id TIMEUUID PRIMARY KEY,
    name TEXT,
    age INT,
    gender TEXT,
    created_at TIMESTAMP
);""")

In [3]:
session.execute(f"INSERT INTO {db_table} (id, name, age, gender, created_at) VALUES (now(), 'John Doe', 30, 'male', currentTimestamp());")
session.execute(f"INSERT INTO {db_table} (id, name, age, gender, created_at) VALUES (now(), 'Jane Smith', 25, 'female', currentTimestamp());")
session.execute(f"INSERT INTO {db_table} (id, name, age, gender, created_at) VALUES (now(), 'Robin E', 29, 'male', currentTimestamp());")

In [4]:
# Query the data
rows = session.execute(f"SELECT * FROM {db_table};")
for i in rows:
    print(i)
    # Extract the timestamp from Cassandra's timeuuid
    print("Datetime:", datetime_from_uuid1(i.id))

Row(id=UUID('746b76a0-cb09-11f1-b703-7b2361473a58'), age=30, created_at=datetime.datetime(2026, 10, 18, 15, 34, 49, 354000), gender='male', name='John Doe')
Datetime: 2026-10-18 15:34:49.354000
Row(id=UUID('746c39f0-cb09-11f1-b703-7b2361473a58'), age=29, created_at=datetime.datetime(2026, 10, 18, 15, 34, 49, 359000), gender='male', name='Robin E')
Datetime: 2026-10-18 15:34:49.359000
Row(id=UUID('746bc4c0-cb09-11f1-b703-7b2361473a58'), age=25, created_at=datetime.datetime(2026, 10, 18, 15, 34, 49, 356000), gender='female', name='Jane Smith')
Datetime: 2026-10-18 15:34:49.356000


In [5]:

spark = (
    SparkSession.builder.appName('SparkCassandraApp')
    .config('spark.jars.packages',
            'com.datastax.spark:spark-cassandra-connector_2.12:3.5.1')
#     .config('spark.sql.extensions',
#             'com.datastax.spark.connector.CassandraSparkExtensions')
#     .config('spark.sql.catalog.mycatalog',
#             'com.datastax.spark.connector.datasource.CassandraCatalog')
    .config('spark.cassandra.connection.host',
            '127.0.0.1')
    .config('spark.cassandra.connection.port',
            '9042')
    .getOrCreate()
)

26/10/18 17:34:50 WARN Utils: Your hostname, hush-kubuntu resolves to a loopback address: 127.0.1.1; using 192.168.1.3 instead (on interface wlo1)
26/10/18 17:34:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/hush/.ivy2/cache
The jars for the packages stored in: /home/hush/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ee387703-8426-40e8-9958-5c2fa6e9b750;1.0
	confs: [default]


:: loading settings :: url = jar:file:/mnt/nvme_storage/projects/NMBU/IND320-hush/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found com.datastax.spark#spark-cassandra-connector_2.12;3.5.1 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.1 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found org.apache.cassandra#java-driver-core-shaded;4.18.1 in central
	found com.datastax.oss#native-protocol;1.5.1 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogram#HdrHistogram;2.1.12 in central
	found org.reactivestreams#reactive-streams;1.0.3 in central
	found org.apache.cassandra#java-driver-mapper-runtime;4.18.1 in central
	found org.apache.cassandra#java-driver-query-builder;4.18.1 in central
	found org.apache.commons#commons-lang3;3.10 in central
	found com.thoughtworks.paranamer#paranamer;2.8 in central
	found org.scala-lang#scala-reflect

In [6]:
spark_df = spark.read.format("org.apache.spark.sql.cassandra").options(table=db_table, keyspace=db_keyspace).load()

spark_df.show()

+--------------------+---+--------------------+------+----------+
|                  id|age|          created_at|gender|      name|
+--------------------+---+--------------------+------+----------+
|746bc4c0-cb09-11f...| 25|2026-10-18 17:34:...|female|Jane Smith|
|746b76a0-cb09-11f...| 30|2026-10-18 17:34:...|  male|  John Doe|
|746c39f0-cb09-11f...| 29|2026-10-18 17:34:...|  male|   Robin E|
+--------------------+---+--------------------+------+----------+



In [7]:
sm = SecretManager()
mongo_username = sm.get_required("mongo_db.username")
mongo_password = sm.get_required("mongo_db.password")
mongo_url = sm.get_required("mongo_db.url").replace("<username>", mongo_username).replace("<password>", mongo_password)

No secrets path provided
Defaulting to: [REDACTED]/IND320-hush/.secrets/secrets.toml


In [8]:
client = MongoClient(mongo_url)
try:
    client.admin.command("ping")
    print("Connected successfully")
    #client.close()

except Exception as e:
    raise Exception(
        "The following error occurred: ", e)

Connected successfully


In [9]:
db = client["ind320"]
collection = db["testing_collection"]

collection.drop() # Drop the collection if it exists to start fresh

collection.insert_one({"name": "John Doe", "age": 30, "gender": "male"})
collection.insert_many([
    {"name": "Jane Smith", "age": 25, "gender": "female"},
    {"name": "Robin E", "age": 29, "gender": "male"}
])

for row in collection.find():
    print(row)

client.close()

{'_id': ObjectId('6ad4e72081b216d17b01d9e2'), 'name': 'John Doe', 'age': 30, 'gender': 'male'}
{'_id': ObjectId('6ad4e72081b216d17b01d9e3'), 'name': 'Jane Smith', 'age': 25, 'gender': 'female'}
{'_id': ObjectId('6ad4e72081b216d17b01d9e4'), 'name': 'Robin E', 'age': 29, 'gender': 'male'}


In [10]:
API = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData"

# init data
req_json = requests.get(API).json()
df_mag = pd.DataFrame(req_json)

df_mag = df_mag.rename(columns={
    "dato_Id": "date",
    "omrType": "area_type",
    "omrnr": "area_number",
    "iso_aar": "iso_year",
    "iso_uke": "iso_week",
    "fyllingsgrad": "water_level",
    "kapasitet_TWh": "twh_cap",
    "fylling_TWh": "twh_fill",
    "neste_Publiseringsdato": "next_publication_date",
    "fyllingsgrad_forrige_uke": "water_level_prev_week",
    "endring_fyllingsgrad": "water_level_change"
})

display(df_mag.head())

,date,area_type,area_number,iso_year,iso_week,water_level,twh_cap,twh_fill,next_publication_date,water_level_prev_week,water_level_change
0,1995-09-03,EL,4,1995,35,0.944721,21.079208,19.913979,0001-01-01T00:00:00,0.936888,0.007833
1,2020-11-15,EL,1,2020,46,0.974361,6.003264,5.849344,2020-11-25T13:00:00,0.997406,-0.023046
2,2011-08-28,EL,4,2011,34,0.786499,21.079208,16.578772,0001-01-01T00:00:00,0.787193,-0.000694
3,1998-07-05,EL,3,1998,27,0.793969,8.920932,7.082947,0001-01-01T00:00:00,0.733812,0.060157
4,2011-03-27,EL,4,2011,12,0.300820,21.079208,6.341054,0001-01-01T00:00:00,0.311122,-0.010301
